## Setting up Transformer Models for Comparison

This notebook will guide you through setting up and comparing three popular transformer models (BERT, DistilBERT, and RoBERTa) on a given Excel dataset. The general steps involve:

1.  **Installing Libraries**: Install `transformers`, `pandas`, and `torch`.
2.  **Loading Data**: Load your Excel file into a pandas DataFrame.
3.  **Data Preprocessing**: Prepare the text data for each model's tokenizer.
4.  **Model Initialization**: Load pre-trained models and their respective tokenizers.
5.  **Task Definition & Fine-tuning/Prediction**: Define a specific NLP task (e.g., text classification) and either fine-tune the models or perform inference.
6.  **Performance Evaluation**: Compare the models' performance using appropriate metrics.

In [1]:
# Install necessary libraries
!pip install transformers pandas torch

# Or if you prefer tensorflow:
# !pip install transformers pandas tensorflow

### 1. Load your Excel Data

Replace `'your_data.xlsx'` with the actual path to your Excel file. Ensure the file contains a column with text data that you want to process.

In [2]:
import pandas as pd

# --- IMPORTANT: Upload your Excel file to Colab or provide the correct path ---
# For example, if your file is named 'my_texts.xlsx' and it's directly in your Colab files:
file_path = 'MCOE 2026 Dataset.xlsx' # <--- CHANGE THIS TO YOUR EXCEL FILE PATH

try:
    df = pd.read_excel(file_path)
    print(f"Successfully loaded data from {file_path}. Shape: {df.shape}")
    display(df.head())
except FileNotFoundError:
    print(f"Error: The file '{file_path}' was not found. Please upload it to Colab or provide the correct path.")
    print("You can upload files by clicking the folder icon on the left sidebar, then the 'Upload to session storage' icon.")
    df = None # Set df to None to avoid errors in subsequent cells if file isn't found

Successfully loaded data from MCOE 2026 Dataset.xlsx. Shape: (3102, 13)


,Premise,Hypothesis,Relationship,ContradictionType,Topic,Subtopic,Sentence Structure,Generated Relationship,Generated Contradiction type,Generated Sentence Structure,Unnamed: 10,Disagreements with Jamie,COPPA without IAN
0,The quantum processor developed by QTech Corp ...,The quantum processor from QTech Corp is limit...,Contradictory,Numerical,Technology,Other,Restatement,NaN,NaN,Same Subject,NaN,381.0,0.173182
1,The new privacy update encrypts all user data ...,The new privacy update fails to encrypt any us...,Contradictory,Negation,Technology,Cybersecurity,Same Subject,NaN,NaN,NaN,NaN,2200.0,NaN
2,The AI model was trained using a dataset consi...,The AI model was trained using diverse dataset...,Contradictory,Attribute,Technology,Other,Restatement,NaN,NaN,Same Subject,NaN,NaN,NaN
3,Tesla’s autopilot system was launched in 2015 ...,Tesla’s autopilot was introduced in 2022 after...,Contradictory,Temporal,Technology,Other,Same Subject,NaN,NaN,NaN,NaN,NaN,NaN
4,The cloud server in Frankfurt ensures data is ...,The cloud server is located in Singapore to re...,Contradictory,Spatial,Technology,Other,Restatement,NaN,NaN,Same Subject,NaN,NaN,NaN


### 2. Initialize Transformer Models and Tokenizers

We will now load the pre-trained tokenizers and models for BERT, DistilBERT, and RoBERTa. For demonstration, we'll use the base versions of these models suitable for sequence classification. If your task is different (e.g., token classification, question answering), you might need to adjust the model type (e.g., `AutoModelForTokenClassification`).

In [3]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

# Define device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Placeholder for number of labels. This will be updated after data preprocessing.
# For now, let's keep it as 2, and we'll ensure it's adjusted after the label creation step.
# Or dynamically set it if we have 'df' available which we don't here.
# We will assume it will be correctly set by the subsequent data preprocessing step
# to the number of unique labels in the 'Relationship' column.
num_unique_labels = 3 # This will be updated once 'df' and 'label' column are processed

# --- BERT ---
bert_model_name = 'bert-base-uncased'
bert_tokenizer = AutoTokenizer.from_pretrained(bert_model_name)
bert_model = AutoModelForSequenceClassification.from_pretrained(bert_model_name, num_labels=num_unique_labels) # Use dynamic num_labels
bert_model.to(device)
print(f"\n{bert_model_name} loaded.")

# --- DistilBERT ---
distilbert_model_name = 'distilbert-base-uncased'
distilbert_tokenizer = AutoTokenizer.from_pretrained(distilbert_model_name)
distilbert_model = AutoModelForSequenceClassification.from_pretrained(distilbert_model_name, num_labels=num_unique_labels)
distilbert_model.to(device)
print(f"\n{distilbert_model_name} loaded.")

# --- RoBERTa ---
roberta_model_name = 'roberta-base'
roberta_tokenizer = AutoTokenizer.from_pretrained(roberta_model_name)
roberta_model = AutoModelForSequenceClassification.from_pretrained(roberta_model_name, num_labels=num_unique_labels)
roberta_model.to(device)
print(f"\n{roberta_model_name} loaded.")

Using device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.weight       | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



roberta-base loaded.


In [4]:
if df is not None:
    # Concatenate 'Premise' and 'Hypothesis' into a single 'text' column
    # Fill any NaN values with an empty string before concatenation
    df['text'] = df['Premise'].fillna('') + ' ' + df['Hypothesis'].fillna('')

    # Display some examples of the combined text
    print("\nExamples of combined text:")
    for i in range(min(5, len(df))):
        print(f"- {df['text'].iloc[i][:150]}...") # Print first 150 characters

    # Inspect the 'Relationship' column for unique labels
    if 'Relationship' in df.columns:
        print("\nUnique labels in 'Relationship' column:")
        print(df['Relationship'].value_counts())

        # Convert 'Relationship' to categorical type and then to numerical codes
        # This will assign a unique integer to each unique string label
        df['label'] = pd.Categorical(df['Relationship']).codes

        # Get the mapping from category name to code
        category_to_code = {category: code for code, category in enumerate(pd.Categorical(df['Relationship']).categories)}
        print("\nCreated 'label' column with mapping:")
        print(category_to_code)
        print(df['label'].value_counts())

        # Update num_unique_labels for model initialization (if df is available globally)
        # Note: This requires rerunning the model initialization cell after this cell.
        global num_unique_labels
        num_unique_labels = len(pd.Categorical(df['Relationship']).categories)
        print(f"\nDetected {num_unique_labels} unique labels. Please ensure you rerun the model initialization cell to update `num_labels`.")

    else:
        print("\nWarning: 'Relationship' column not found. You will need to define your 'label' column manually for classification.")
        df['label'] = None
else:
    print("DataFrame 'df' is not available. Please ensure the Excel file was loaded correctly.")


Examples of combined text:
- The quantum processor developed by QTech Corp can perform over a million operations per second. The quantum processor from QTech Corp is limited to ju...
- The new privacy update encrypts all user data end-to-end using a zero-knowledge protocol. The new privacy update fails to encrypt any user data....
- The AI model was trained using a dataset consisting exclusively of financial news articles. The AI model was trained using diverse datasets including ...
- Tesla’s autopilot system was launched in 2015 and has undergone regular updates since then. Tesla’s autopilot was introduced in 2022 after years of de...
- The cloud server in Frankfurt ensures data is stored within the EU for regulatory compliance. The cloud server is located in Singapore to reduce laten...

Unique labels in 'Relationship' column:
Relationship
Contradictory    1178
Entailing        1087
Neutral           837
Name: count, dtype: int64

Created 'label' column with mapping:
{'Contradictor

### 4. Tokenization

Now we'll tokenize the 'text' column using each model's respective tokenizer. We'll also split the data into training and validation sets.

In [5]:
from sklearn.model_selection import train_test_split

# Ensure num_unique_labels is correctly set based on the 'label' column
# This check is crucial if the user reran the preprocessing cell separately.
if 'df' in globals() and df is not None and 'label' in df.columns:
    current_num_labels = len(df['label'].unique())
    print(f"Detected {current_num_labels} unique labels in the dataset.")
    # Optionally, re-initialize models if num_unique_labels has changed from last run of cell 853a0a79
    # This part can be more complex if models need full re-init. For simplicity, we assume
    # the user has re-run the model initialization cell if prompted.
else:
    print("Warning: 'df' or 'label' column not found. Tokenization may fail or use default num_labels (3).")
    current_num_labels = num_unique_labels # Fallback to global placeholder if df is not ready

if df is not None and 'text' in df.columns and 'label' in df.columns and df['label'].notna().any():
    # Drop rows where text or label is missing for training
    df_cleaned = df.dropna(subset=['text', 'label']).copy()

    # Convert labels to integers
    df_cleaned['label'] = df_cleaned['label'].astype(int)

    # Split data into training and validation sets
    train_texts, val_texts, train_labels, val_labels = train_test_split(
        df_cleaned['text'].tolist(),
        df_cleaned['label'].tolist(),
        test_size=0.2, # 20% for validation
        random_state=42,
        stratify=df_cleaned['label'].tolist() # Ensure balanced classes in splits
    )

    print(f"\nTraining samples: {len(train_texts)}")
    print(f"Validation samples: {len(val_texts)}")

    # Tokenize data for each model
    def tokenize_data(tokenizer, texts):
        return tokenizer(texts, padding=True, truncation=True, return_tensors='pt')

    print("\nTokenizing data for BERT...")
    train_encodings_bert = tokenize_data(bert_tokenizer, train_texts)
    val_encodings_bert = tokenize_data(bert_tokenizer, val_texts)

    print("Tokenizing data for DistilBERT...")
    train_encodings_distilbert = tokenize_data(distilbert_tokenizer, train_texts)
    val_encodings_distilbert = tokenize_data(distilbert_tokenizer, val_texts)

    print("Tokenizing data for RoBERTa...")
    train_encodings_roberta = tokenize_data(roberta_tokenizer, train_texts)
    val_encodings_roberta = tokenize_data(roberta_tokenizer, val_texts)

    print("Tokenization complete.")
else:
    print("Cannot proceed with tokenization: DataFrame 'df' is not available, or 'text'/'label' columns are missing/empty. Please ensure the Excel file was loaded and preprocessed correctly.")


Detected 3 unique labels in the dataset.

Training samples: 2481
Validation samples: 621

Tokenizing data for BERT...
Tokenizing data for DistilBERT...
Tokenizing data for RoBERTa...
Tokenization complete.


### 5. Create PyTorch Datasets and DataLoaders

To train and evaluate our models, we need to convert our tokenized data and labels into PyTorch `Dataset` objects and then wrap them with `DataLoader`s. This helps in efficient batching and shuffling of data.

In [6]:
from torch.utils.data import Dataset, DataLoader

class TextDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: val[idx].clone().detach() for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

if 'train_encodings_bert' in locals():
    # Create datasets
    train_dataset_bert = TextDataset(train_encodings_bert, train_labels)
    val_dataset_bert = TextDataset(val_encodings_bert, val_labels)

    train_dataset_distilbert = TextDataset(train_encodings_distilbert, train_labels)
    val_dataset_distilbert = TextDataset(val_encodings_distilbert, val_labels)

    train_dataset_roberta = TextDataset(train_encodings_roberta, train_labels)
    val_dataset_roberta = TextDataset(val_encodings_roberta, val_labels)

    # Create DataLoaders
    batch_size = 16 # You can adjust this based on your GPU memory and preference

    train_loader_bert = DataLoader(train_dataset_bert, batch_size=batch_size, shuffle=True)
    val_loader_bert = DataLoader(val_dataset_bert, batch_size=batch_size, shuffle=False)

    train_loader_distilbert = DataLoader(train_dataset_distilbert, batch_size=batch_size, shuffle=True)
    val_loader_distilbert = DataLoader(val_dataset_distilbert, batch_size=batch_size, shuffle=False)

    train_loader_roberta = DataLoader(train_dataset_roberta, batch_size=batch_size, shuffle=True)
    val_loader_roberta = DataLoader(val_dataset_roberta, batch_size=batch_size, shuffle=False)

    print("PyTorch Datasets and DataLoaders created for all models.")
else:
    print("Cannot create Datasets/DataLoaders: Tokenized data is not available. Please ensure previous steps ran successfully.")

PyTorch Datasets and DataLoaders created for all models.


### 6. Training and Evaluation Strategy

Now that the data is prepared, we can proceed with training and evaluating the models. There are several approaches you can take:

**Option A: Fine-tuning with `Trainer` API (Recommended)**

The `transformers` library provides a high-level `Trainer` API that simplifies the fine-tuning process significantly. This is generally the easiest and most robust way to fine-tune Hugging Face models.

**Option B: Manual Training Loop**

If you need more control over the training process, you can write a custom PyTorch training loop. This involves iterating through `DataLoader`s, performing forward passes, calculating loss, backpropagating, and updating weights.

**Option C: Zero-shot Classification (if applicable)**

If your task is suitable for zero-shot classification (i.e., classifying text into categories without explicit training data for those categories), you could use a zero-shot classification pipeline. However, since you have an Excel dataset, fine-tuning is usually more appropriate for better performance on your specific data.

---

**For this walkthrough, I will proceed with Option A (Fine-tuning with `Trainer` API) as it's the most common and efficient way to fine-tune these models.** We will define a training function that can be reused for each model.

### 7. Fine-tuning Models with `Trainer` API

We will now fine-tune each of the models using the `Trainer` API from the `transformers` library. This involves:

1.  **Defining `TrainingArguments`**: Specifies training hyperparameters.
2.  **Defining `compute_metrics` function**: Calculates evaluation metrics (e.g., accuracy, F1-score).
3.  **Initializing `Trainer`**: Combines the model, arguments, datasets, and metrics.
4.  **Training the model**: Calling the `train()` method.
5.  **Evaluating the model**: Calling the `evaluate()` method.

In [7]:
from transformers import TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
import numpy as np

def compute_metrics(p):
    predictions = np.argmax(p.predictions, axis=1)
    accuracy = accuracy_score(p.label_ids, predictions)
    f1 = f1_score(p.label_ids, predictions, average='weighted') # 'weighted' is good for imbalanced classes
    precision = precision_score(p.label_ids, predictions, average='weighted', zero_division=0)
    recall = recall_score(p.label_ids, predictions, average='weighted', zero_division=0)
    return {
        'accuracy': accuracy,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

# Check if datasets and loaders were successfully created
if 'train_dataset_bert' not in locals():
    print("Training cannot proceed: Datasets/DataLoaders not available. Please ensure previous steps ran successfully.")
else:
    # --- Training Arguments ---
    # You can customize these arguments based on your needs and resources
    training_args = TrainingArguments(
        output_dir='./results',
        num_train_epochs=3,               # Total number of training epochs
        per_device_train_batch_size=16,  # Batch size per GPU/CPU for training
        per_device_eval_batch_size=16,   # Batch size per GPU/CPU for evaluation
        warmup_steps=500,                 # Number of warmup steps for learning rate scheduler
        weight_decay=0.01,                # Strength of weight decay
        logging_dir='./logs',
        logging_steps=100,                # Log every x updates steps
        eval_strategy="epoch",      # Evaluate every epoch
        save_strategy="epoch",            # Save model every epoch
        load_best_model_at_end=True,      # Load the best model after training
        metric_for_best_model="f1",       # Metric to use to compare models
        greater_is_better=True,
        report_to="none"                  # Don't report to any online service like Weights & Biases
    )

    models_to_train = {
        "BERT": {"model": bert_model, "tokenizer": bert_tokenizer, "train_dataset": train_dataset_bert, "eval_dataset": val_dataset_bert},
        "DistilBERT": {"model": distilbert_model, "tokenizer": distilbert_tokenizer, "train_dataset": train_dataset_distilbert, "eval_dataset": val_dataset_distilbert},
        "RoBERTa": {"model": roberta_model, "tokenizer": roberta_tokenizer, "train_dataset": train_dataset_roberta, "eval_dataset": val_dataset_roberta}
    }

    evaluation_results = {}

    for model_name, data in models_to_train.items():
        print(f"\n--- Training and Evaluating {model_name} ---")
        trainer = Trainer(
            model=data["model"],
            args=training_args,
            train_dataset=data["train_dataset"],
            eval_dataset=data["eval_dataset"],
            compute_metrics=compute_metrics
        )

        # Train the model
        trainer.train()

        # Evaluate the model
        eval_metrics = trainer.evaluate()
        evaluation_results[model_name] = eval_metrics
        print(f"Evaluation results for {model_name}: {eval_metrics}")

    print("\n--- All Models Trained and Evaluated ---")
    print("Summary of Evaluation Results:")
    for model_name, metrics in evaluation_results.items():
        print(f"\n{model_name}:")
        for metric, value in metrics.items():
            print(f"  {metric}: {value:.4f}")

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.



--- Training and Evaluating BERT ---


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,1.085605,0.847294,0.605475,0.586859,0.618929,0.605475
2,0.671572,0.626712,0.739130,0.737699,0.756329,0.739130
3,0.454170,0.621862,0.761675,0.756494,0.782660,0.761675


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch,Accuracy,F1,Precision,Recall
0.361835,0.477200,3,0.834138,0.833328,0.842371,0.834138


Evaluation results for RoBERTa: {'eval_loss': 0.4772002696990967, 'eval_accuracy': 0.8341384863123994, 'eval_f1': 0.8333276581024028, 'eval_precision': 0.8423709649218378, 'eval_recall': 0.8341384863123994}

--- All Models Trained and Evaluated ---
Summary of Evaluation Results:

BERT:
  eval_loss: 0.6219
  eval_accuracy: 0.7617
  eval_f1: 0.7565
  eval_precision: 0.7827
  eval_recall: 0.7617

DistilBERT:
  eval_loss: 0.6236
  eval_accuracy: 0.7359
  eval_f1: 0.7312
  eval_precision: 0.7612
  eval_recall: 0.7359

RoBERTa:
  eval_loss: 0.4772
  eval_accuracy: 0.8341
  eval_f1: 0.8333
  eval_precision: 0.8424
  eval_recall: 0.8341


### 8. Conclusion and Comparison

Based on the evaluation results above, you can compare the performance of BERT, DistilBERT, and RoBERTa on your specific dataset. Key metrics to consider are:

*   **Accuracy**: Overall correctness of predictions.
*   **F1-Score**: Harmonic mean of precision and recall, good for imbalanced datasets.
*   **Precision**: Proportion of positive identifications that were actually correct.
*   **Recall**: Proportion of actual positives that were identified correctly.

DistilBERT is generally faster and smaller than BERT, but may have slightly lower performance. RoBERTa often outperforms BERT due to its training methodology but is typically similar in size and speed to BERT. The 'best' model will depend on your specific task's requirements for performance, speed, and resource usage.